# Geracao de perfis sinteticos brasileiros

Este notebook demonstra o pipeline modular do pacote `synthetic_br_profiles_gan`. Ele nao reimplementa geradores, validadores, modelos ou metricas; todas as etapas chamam codigo de `src/`.

Os dados gerados sao sinteticos. O projeto nao consulta bases oficiais, Receita Federal, cartorios, operadoras ou qualquer fonte externa para verificar documentos ou pessoas reais.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from synthetic_br_profiles_gan.config import deep_merge
from synthetic_br_profiles_gan.pipeline import DEFAULT_PIPELINE_CONFIG, run_pipeline


## Execucao pequena

A configuracao abaixo usa o baseline programatico para uma execucao rapida. Para comparar modelos, altere `model` para `simple_gan` ou `ctgan` e instale o extra correspondente.

In [ ]:
config = deep_merge(
    DEFAULT_PIPELINE_CONFIG,
    {
        'artifacts_root': str(PROJECT_ROOT / 'artifacts'),
        'seed': 41,
        'reference_date': '2026-07-26',
        'model': 'programmatic',
        'calibration': {'seed': 41, 'num_rows': 500, 'holdout_fraction': 0.2},
        'models': {'programmatic': {'seed': 100044}},
        'generation': {'rows': 100, 'batch_size': 128, 'max_batches': 5, 'date_format': '%Y-%m-%d'},
        'export': {'xlsx': True},
    },
)

result = run_pipeline(config=config, model_name='programmatic')
result['run_id'], result['status'], result['paths']['manifest']


## Amostra e validacao

In [ ]:
result['dataset'].head()


In [ ]:
result['validation']


## Metricas de qualidade e privacidade

As metricas de privacidade sao indicadores de risco de memorizacao. Elas nao provam anonimizacao.

In [ ]:
quality = result['quality_gates']
privacy = result['evaluation']['privacy']
{
    'status': quality['status'],
    'failures': quality['failures'],
    'exact_train_match_rate': privacy['exact_train_match_rate'],
    'duplicate_row_rate': privacy['duplicate_row_rate'],
    'unique_combination_rate': privacy['unique_combination_rate'],
}


## Comparacao entre modelos

Use os mesmos dados de calibracao, a mesma seed e o mesmo conjunto de quality gates para comparar `programmatic`, `simple_gan` e `ctgan`. A CTGAN usa a biblioteca standalone `ctgan` e recebe explicitamente as colunas categoricas/discretas.